In [ ]:
import huggingface_hub
import tokenizers
import torch
import torch.nn as nn

# 1. 模型结构

In [ ]:
# 原始的FFN
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False, dytpe=cfg["dtype"])
        self.w2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False, dytpe=cfg["dtype"])
        self.w3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False, dytpe=cfg["dtype"])
    
    def forward(self, x):
        x = self.w1(x) * nn.functional.silu(self.w2(x))
        return self.w3(x)

In [ ]:
# 将FFN改造为MoE结构
class MoEFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # 激活专家数(与deepseek-v3的区别是，没有共享专家)
        self.num_experts_per_tok = cfg["num_experts_per_tok"]
        # 总专家数
        self.num_experts = cfg["num_experts"]
        # 门控层
        self.gate = nn.Linear(cfg["emb_dim"], cfg["num_experts"], bias=False, dtype=cfg["dtype"])

        # meta device to reduce memory pressure when initializing the model before loading weights
        meta_device = torch.device("meta")

         # 专家层(类似原始FFN)
        self.fc1 = nn.ModuleList([
            nn.Linear(
               cfg["emb_dim"], cfg["moe_intermediate_size"],
               bias=False, dtype=cfg["dtype"], device=meta_device)
               for _ in range(cfg["num_experts"])]
        )
        self.fc2 = nn.ModuleList([
            nn.Linear(
               cfg["emb_dim"], cfg["moe_intermediate_size"],
               bias=False, dtype=cfg["dtype"], device=meta_device)
               for _ in range(cfg["num_experts"])]
        )
        self.fc3 = nn.ModuleList([
            nn.Linear(
               cfg["moe_intermediate_size"], cfg["emb_dim"],
               bias=False, dtype=cfg["dtype"], device=meta_device)
               for _ in range(cfg["num_experts"])]
        )
    
    def forward(self, x):
        b, seq_len, embed_dim = x.shape
        # shape = (b, seq_len, num_experts)
        scores = self.gate(x)
        # 取topk
        topk_scores, topk_indices = torch.topk(scores, self.num_experts_per_tok, dim=-1)
        # 算各个专家的probs
        topk_probs = torch.softmax(topk_scores, dim=-1)

        # 计算各个专家的输出
        expert_outputs = []
        for e in range(self.num_experts):
            hidden = torch.nn.functional.silu(self.fc[e](x)) * self.fc2[e](x)
            # shape = (b, seq_len, embed_dim)
            out = self.fc3[e](hidden)
            # shape = (b, seq_len, 1, embed_dim)
            expert_outputs.append(out.unsqueeze(-2))
        # shape = (b, seq_len, num_experts, embed_dim)
        expert_outputs = torch.cat(expert_outputs, dim=-2)

        gating_probs = torch.zeros_like(scores)

        for i in range(self.num_experts_per_tok):
            # i:i+1的取法，可以保留长度为1的最后一维
            # shape = (b, seq_len, 1)
            indices = topk_indices[..., i:i+1]
            prob = topk_probs[..., i:i+1]
            # 将probs原地填充回shape = (b, seq_len, num_experts)
            gating_probs.scatter_(dim=-1, index=indices, src=prob)
        # shape = (b, seq_len, num_experts)
        gating_probs = gating_probs.unsqueeze(-1)

        # 加权专家输出
        y = (gating_probs * expert_outputs).sum(dim=-2)
        return y

    # def forward_bak(self, x):
    #     scores = self.gate(x)
    #     topk_scores, topk_indices = torch.topk(scores, self.num_experts_per_tok, dim=-1)
    #     topk_probs = torch.softmax(topk_scores, dim=-1)
    #     y = torch.zeros_like(x)

    #     for i in range(self.num_experts_per_tok):
    #         expert_indices = topk_indices[..., i]
    #         prob = topk_probs[..., i].unsqueeze(-1)

    #         for e in range(self.num_experts):
    #             mask = (expert_indices == e)
    #             if mask.any():
    #                 selected = x[mask]
    #                 out = self.fc3[e](nn.functional.silu(self.fc1[e](selected)) * self.fc2[e](selected))
    #                 y[mask] += prob[mask] * out
    #     return y

In [ ]:
# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None
    
    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale
        
        if self.shift is not None:
            norm_x = norm_x + self.shift
        
        return norm_x.to(input_dtype)

In [ ]:
# RoPE
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angels
    angles = positions[:, None] * inv_freq[None, :]

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head dimension must be even"

    x1 = x[..., : head_dim // 2]
    x2 = x[..., head_dim // 2 :]

    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

In [ ]:
# GQA
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads
        
        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim)
            self.k_norm = RMSNorm(head_dim)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # qkv
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # qk norm
        if self.q_norm is not None:
            queries = self.q_norm(queries)
        if self.k_norm is not None:
            keys = self.k_norm(keys)
        
        # rope
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # expand
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # attn
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        att_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        # context vec
        context = (att_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)

In [ ]:
# transformer
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            head_dim=cfg["head_dim"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )

        #########################################################
        ### 与dense模型的区别是，用MoE结构替换原始FFN
        ### 另外两种MoE采用的方式分别是：
        ### 1. 前n层用ffn，后续用moe（deepseek-v3采用）
        ### 2. ffn和moe交替使用（Qwen3 235B-A22B采用）
        #########################################################
        if cfg["num_experts"] > 0:
            self.ff = MoEFeedForward(cfg)
        else:
            self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(emb_dim=cfg["emb_dim"])
        self.norm2 = RMSNorm(emb_dim=cfg["emb_dim"])

    def forward(self, x, mask, cos, sin):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x) 
        x = shortcut + x

        return x

In [ ]:
# qwen3 moe
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # params
        self.head_dim = cfg.get("head_dim", None)
        if self.head_dim is None:
            assert cfg["emb_dim"] % cfg["n_heads"] == 0, "`emb_dim` must be divisible by `num_heads` if `head_dim` is not set"
            self.head_dim = cfg["emb_dim"] // cfg["n_heads"]
        self.cfg = cfg

        # emb
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        # transformer blocks
        self.trf_blocks = nn.ModuleList([
            TransformerBlock(cfg) for _ in range(cfg["n_layers"])
        ])

        # out_head
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # rope
        cos, sin = compute_rope_params(
            self.head_dim,
            theta_base=cfg["rope_base"], 
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    
    def forward(self, in_idx):
        b, seq_len = in_idx.shape

        # emb
        x = self.tok_emb(in_idx)

        # mask
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1)

        # trf
        for trf in self.trf_blocks:
            x = trf(x, mask, self.cos, self.sin)
        
        # logits
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits
